In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('../data/raw/application_train.csv')

# Basic checks - always do this first with any new dataset
print("Shape (rows, columns):", df.shape)
print("\nColumn names (first 20):")
print(df.columns[:20].tolist())
print("\nFirst 5 rows:")
df.head()

: 

In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('../data/raw/application_train.csv')

# Basic checks - always do this first with any new dataset
print("Shape (rows, columns):", df.shape)
print("\nColumn names (first 20):")
print(df.columns[:20].tolist())
print("\nFirst 5 rows:")
df.head()

Shape (rows, columns): (307511, 122)

Column names (first 20):
['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION']

First 5 rows:


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [2]:
# TARGET: 1 = client had payment difficulties (defaulted), 0 = paid on time
print("Target value counts:")
print(df['TARGET'].value_counts())

print("\nDefault rate (%):")
print(df['TARGET'].value_counts(normalize=True) * 100)

Target value counts:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Default rate (%):
TARGET
0    91.927118
1     8.072882
Name: proportion, dtype: float64


In [3]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_summary = missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_pct', ascending=False)

print(f"Columns with missing values: {len(missing_summary)} out of {df.shape[1]}")
missing_summary.head(20)

Columns with missing values: 67 out of 122


,missing_count,missing_pct
COMMONAREA_MEDI,214865,69.872297
COMMONAREA_MODE,214865,69.872297
COMMONAREA_AVG,214865,69.872297
NONLIVINGAPARTMENTS_MODE,213514,69.432963
NONLIVINGAPARTMENTS_MEDI,213514,69.432963
NONLIVINGAPARTMENTS_AVG,213514,69.432963
FONDKAPREMONT_MODE,210295,68.386172
LIVINGAPARTMENTS_AVG,210199,68.354953
LIVINGAPARTMENTS_MEDI,210199,68.354953
LIVINGAPARTMENTS_MODE,210199,68.354953


In [4]:
print("Gender distribution:")
print(df['CODE_GENDER'].value_counts())

print("\nAge range (in years, converted from days):")
print((-df['DAYS_BIRTH'] / 365).describe())


Gender distribution:
CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64

Age range (in years, converted from days):
count    307511.000000
mean         43.936973
std          11.956133
min          20.517808
25%          34.008219
50%          43.150685
75%          53.923288
max          69.120548
Name: DAYS_BIRTH, dtype: float64


In [5]:
# First look at fairness - does default rate differ by gender?
gender_default = df.groupby('CODE_GENDER')['TARGET'].agg(['mean', 'count'])
gender_default.columns = ['default_rate', 'num_applicants']
gender_default['default_rate_pct'] = gender_default['default_rate'] * 100
print(gender_default)

             default_rate  num_applicants  default_rate_pct
CODE_GENDER                                                
F                0.069993          202448          6.999328
M                0.101419          105059         10.141920
XNA              0.000000               4          0.000000


## Key EDA Insight: Gender & Default Rate
- Female applicants: 7.0% default rate (202,448 applicants)
- Male applicants: 10.1% default rate (105,059 applicants)
- Men default at ~1.45x the rate of women in raw data
- This baseline will be compared against model predictions during the Week 3 fairness audit

In [6]:
# Bucket age into groups for fairness check
df['age_years'] = -df['DAYS_BIRTH'] / 365
df['age_group'] = pd.cut(df['age_years'], bins=[18, 25, 35, 45, 55, 65, 100], 
                           labels=['18-25', '26-35', '36-45', '46-55', '56-65', '65+'])

age_default = df.groupby('age_group', observed=True)['TARGET'].agg(['mean', 'count'])
age_default.columns = ['default_rate', 'num_applicants']
age_default['default_rate_pct'] = age_default['default_rate'] * 100
print(age_default)

           default_rate  num_applicants  default_rate_pct
age_group                                                
18-25          0.123036           12159         12.303643
26-35          0.106733           72302         10.673287
36-45          0.084047           84274          8.404727
46-55          0.070580           70077          7.057951
56-65          0.054145           60596          5.414549
65+            0.037270            8103          3.727015


C:\Users\Shree\AppData\Local\Temp\ipykernel_18544\2468952390.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['age_years'] = -df['DAYS_BIRTH'] / 365
C:\Users\Shree\AppData\Local\Temp\ipykernel_18544\2468952390.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['age_group'] = pd.cut(df['age_years'], bins=[18, 25, 35, 45, 55, 65, 100],


## Key EDA Insight: Age & Default Rate
- Default rate steadily decreases with age: 12.3% (18-25) down to 3.7% (65+)
- ~3.3x gap between youngest and oldest applicants
- Younger applicants are inherently higher-risk (less credit history, income stability)
- Need to check in Week 3 whether the model's errors disproportionately affect younger applicants, since age-based patterns can be legally sensitive in some contexts (age discrimination in lending)

In [7]:
# Correlation of numeric features with TARGET
numeric_df = df.select_dtypes(include=['int64', 'float64'])
correlations = numeric_df.corr()['TARGET'].sort_values()

print("Top 10 features negatively correlated with default (lower risk):")
print(correlations.head(10))

print("\nTop 10 features positively correlated with default (higher risk):")
print(correlations.tail(11)[:-1])  # excluding TARGET itself

Top 10 features negatively correlated with default (lower risk):
EXT_SOURCE_3                 -0.178919
EXT_SOURCE_2                 -0.160472
EXT_SOURCE_1                 -0.155317
age_years                    -0.078239
DAYS_EMPLOYED                -0.044932
FLOORSMAX_AVG                -0.044003
FLOORSMAX_MEDI               -0.043768
FLOORSMAX_MODE               -0.043226
AMT_GOODS_PRICE              -0.039645
REGION_POPULATION_RELATIVE   -0.037227
Name: TARGET, dtype: float64

Top 10 features positively correlated with default (higher risk):
DAYS_REGISTRATION              0.041975
FLAG_DOCUMENT_3                0.044346
REG_CITY_NOT_LIVE_CITY         0.044395
FLAG_EMP_PHONE                 0.045982
REG_CITY_NOT_WORK_CITY         0.050994
DAYS_ID_PUBLISH                0.051457
DAYS_LAST_PHONE_CHANGE         0.055218
REGION_RATING_CLIENT           0.058899
REGION_RATING_CLIENT_W_CITY    0.060893
DAYS_BIRTH                     0.078239
Name: TARGET, dtype: float64


In [8]:
# Check missing % specifically in our top predictors
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
print(df[ext_cols].isnull().mean() * 100)

EXT_SOURCE_1    56.381073
EXT_SOURCE_2     0.214626
EXT_SOURCE_3    19.825307
dtype: float64


In [9]:
# Only 4 rows - safe to drop, won't affect the model meaningfully
print("Before:", df.shape)
df = df[df['CODE_GENDER'] != 'XNA']
print("After:", df.shape)

Before: (307511, 124)
After: (307507, 124)


In [10]:
# Strategy: 
# - For EXT_SOURCE columns (important, numeric): fill missing with median
# - For columns missing >50%: drop them (too unreliable to use)
# - For remaining numeric columns: fill with median
# - For remaining categorical columns: fill with 'Unknown'

# Drop columns missing more than 50% of data
missing_pct = df.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 50].index.tolist()
print(f"Dropping {len(cols_to_drop)} columns with >50% missing data")

df_clean = df.drop(columns=cols_to_drop)
print("Shape after dropping high-missing columns:", df_clean.shape)

Dropping 41 columns with >50% missing data
Shape after dropping high-missing columns: (307507, 83)


In [11]:
# Fill numeric columns with median
numeric_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Fill categorical columns with 'Unknown'
categorical_cols = df_clean.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna('Unknown')

# Verify no missing values remain
print("Remaining missing values:", df_clean.isnull().sum().sum())

C:\Users\Shree\AppData\Local\Temp\ipykernel_18544\2145207665.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_clean.select_dtypes(include=['object']).columns


Remaining missing values: 0


In [12]:
# Check how many categorical columns we have and their unique value counts
categorical_cols = df_clean.select_dtypes(include=['object']).columns
print(f"Number of categorical columns: {len(categorical_cols)}")
print(df_clean[categorical_cols].nunique().sort_values(ascending=False))

C:\Users\Shree\AppData\Local\Temp\ipykernel_18544\3565470447.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_clean.select_dtypes(include=['object']).columns


Number of categorical columns: 13
ORGANIZATION_TYPE             58
OCCUPATION_TYPE               19
NAME_INCOME_TYPE               8
NAME_TYPE_SUITE                8
WEEKDAY_APPR_PROCESS_START     7
NAME_FAMILY_STATUS             6
NAME_HOUSING_TYPE              6
NAME_EDUCATION_TYPE            5
EMERGENCYSTATE_MODE            3
FLAG_OWN_REALTY                2
FLAG_OWN_CAR                   2
CODE_GENDER                    2
NAME_CONTRACT_TYPE             2
dtype: int64


In [13]:
# Save original CODE_GENDER for fairness audit later (before it gets one-hot encoded)
protected_attributes = df_clean[['CODE_GENDER']].copy()
protected_attributes['age_group'] = df_clean['age_group']  # already created earlier

print(protected_attributes.head())
print("\nShape:", protected_attributes.shape)

  CODE_GENDER age_group
0           M     26-35
1           F     46-55
2           M     46-55
3           F     46-55
4           M     46-55

Shape: (307507, 2)


In [14]:
# One-hot encode all categorical columns
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

print("Shape before encoding:", df_clean.shape)
print("Shape after encoding:", df_encoded.shape)

Shape before encoding: (307507, 83)
Shape after encoding: (307507, 185)


In [15]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
# Drop TARGET (what we're predicting) and SK_ID_CURR (just an ID, not useful for prediction)
X = df_encoded.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df_encoded['TARGET']

# Split into train and test sets (80% train, 20% test)
# stratify=y ensures both sets keep the same 8% default rate - important since data is imbalanced
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("\nDefault rate in train:", y_train.mean())
print("Default rate in test:", y_test.mean())

Training set shape: (246005, 183)
Test set shape: (61502, 183)

Default rate in train: 0.08073006646206378
Default rate in test: 0.08072908198107379


In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Train a basic Logistic Regression model
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)

# Predict on the test set
y_pred = log_reg.predict(X_test)
y_pred_proba = log_reg.predict_proba(X_test)[:, 1]  # probability of default

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

ValueError: Cannot cast str dtype to float64

In [18]:
# age_group and age_years snuck into X as non-numeric / redundant columns
# We already saved age_group separately in protected_attributes for the fairness audit
# age_years is redundant with DAYS_BIRTH which is already in the data

print(X.dtypes[X.dtypes == 'object'])  # check if anything else is lurking
print(X.dtypes[X.dtypes.astype(str).str.contains('category')])  # check for category dtype columns

Series([], dtype: object)
age_group    category
dtype: object


In [19]:
# Drop the problematic column(s) from X
cols_to_remove = ['age_group']  # add 'age_years' too if it shows up in the check above
X_train = X_train.drop(columns=[c for c in cols_to_remove if c in X_train.columns])
X_test = X_test.drop(columns=[c for c in cols_to_remove if c in X_test.columns])

print("X_train shape after fix:", X_train.shape)
print("X_test shape after fix:", X_test.shape)

X_train shape after fix: (246005, 182)
X_test shape after fix: (61502, 182)


In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)
y_pred_proba = log_reg.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

c:\Users\Shree\Desktop\loan-fairness-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy: 0.9192383987512601
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
ROC-AUC: 0.6287390333961446


In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [22]:
# class_weight='balanced' tells the model to pay more attention to the minority class (defaulters)
log_reg_balanced = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
log_reg_balanced.fit(X_train_scaled, y_train)

y_pred_balanced = log_reg_balanced.predict(X_test_scaled)
y_pred_proba_balanced = log_reg_balanced.predict_proba(X_test_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred_balanced))
print("Precision:", precision_score(y_test, y_pred_balanced))
print("Recall:", recall_score(y_test, y_pred_balanced))
print("F1 Score:", f1_score(y_test, y_pred_balanced))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba_balanced))

Accuracy: 0.689115801112159
Precision: 0.16056783847297493
Recall: 0.6743202416918429
F1 Score: 0.2593740316083049
ROC-AUC: 0.7458767753281406
